In [1]:
%cd ../../
%load_ext dotenv
%dotenv

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [2]:
from pathlib import Path

import polars as pl
import pandas as pd

# Load data

In [ ]:
schema = {
    'date': pl.Date,
    'meal': pl.String,
    'meal_type': pl.String,
    'meal_code': pl.Int64,
    'restaurant': pl.String,
    'co2': pl.Float64,
    'pcs': pl.Int64,
    'src': pl.String,
}

pos_intermediate = pl.read_excel("data/inter/intermediate/*.xlsx", schema_overrides=schema)

pos_intermediate.head()

date,meal,meal_type,meal_code,restaurant,co2,pcs
date,str,str,i64,str,f64,i64
2025-09-01,"""Vegaani, Take away""",null,10093,"""570 Viikuna""",null,1
2025-09-01,"""Bar Myöhä Bbq-seitanbowl""",null,200006,"""570 Viikuna""",null,1
2025-09-01,"""Take away ruoka""",null,3215,"""570 Viikuna""",null,1
2025-09-01,"""Kasvisjalapenonugetteja ja tom…",null,9043,"""570 Viikuna""",null,1
2025-09-01,"""Pizza, Kasvis""",null,1009,"""570 Viikuna""",null,2


# Extract new dishes

In [6]:
path = "data/processed/dim_meal_types.xlsx"
dim_meal_types = pl.read_excel(path)
dim_meal_types.tail()

meal_type_id,meal_type,meal_type_en
i64,str,str
3,"""Kana""","""chicken"""
4,"""Vegaani""","""vegan"""
5,"""Kasvis""","""vegetarian"""
6,"""Buffet""","""buffet"""
7,"""Not Mapped""","""not_mapped"""


In [7]:
path = "data/processed/dim_restaurants.xlsx"

dim_restaurants = pl.read_excel(path)

dim_restaurants.head()

restaurant_id,restaurant,restaurant_short
i64,str,str
1,"""600 Chemicum""","""che"""
2,"""620 Exactum""","""exa"""
3,"""610 Physicum""","""phy"""
4,"""570 Viikuna""","""vik"""


In [8]:
path = "data/processed/dim_meals.parquet"

dim_meals = pl.read_parquet(path)
dim_meals.head()

id,meal_codes,names,restaurants,meal_type,schoolyear,attributes,co2,src
i64,list[i64],list[str],list[i64],i64,str,list[str],f32,list[str]
0,[90000000],"[""Kikhernetaginea& syysomenajogurttia""]",[1],5,"""23-24""",[],0.41,"[""pos_Jan23-Oct24""]"
1,[90000001],"[""Rapea Meiramikana""]","[1, 4]",3,"""23-24""",[],1.26,"[""pos_Jan23-Oct24""]"
2,[7203],"[""Kasvismuhennos Caponata""]","[1, 2, 4]",4,"""23-24""",[],0.42,"[""pos_Jan23-Oct24"", ""menus_meals""]"
3,[9058],"[""TexMex-siemenpyöryköitä ja Arrabiattakastiketta"", ""TexMex-siemenpyöryköitä ja Arrabiattakastiket""]","[1, 2, 4]",4,"""24-25""","[""gluten_free"", ""vegan-kpl""]",0.56,"[""pos_Jan23-Oct24"", ""menus_meals"", … ""menus_week1-6""]"
4,[6877],"[""Kasvisjalapenonuggetteja, tomaattisalsaa"", ""Kasvis-jalapnuget ja tomatsals""]","[1, 2, 4]",4,"""24-25""",[],0.44,"[""pos_Jan23-Oct24"", ""menus_meals"", ""pos_Nov24-Mar25""]"


In [9]:
dishes_new = (
    pos_intermediate

    .join(dim_restaurants, on='restaurant', how='left')
    .drop('restaurant', 'restaurant_short')

    # Group meal
    .group_by('meal_code', 'meal')
    .agg(pl.concat_list('restaurant_id').flatten().alias('restaurants'))
    .with_columns(pl.col('restaurants').list.unique())

    .join(
        dim_meals.select('id', pl.col('meal_codes').alias('meal_code')).explode('meal_code'),
        on='meal_code',
        how='anti'
    )
)

dishes_new.head()

meal_code,meal,restaurants
i64,str,list[i64]
9313,"""Luomutofua pestokastikkeessa""","[2, 4]"
9308,"""Kreikkaista pähkinä-pinaattika…","[1, 2, 4]"
9331,"""Kebabmauste-kasvispyörykät & t…",[4]
9339,"""Afrikkalainen tomaattinen tagi…",[1]
9328,"""Härkis-BBQ-kastiketta""","[2, 4]"


Supplement fields for list of new dishes

In [10]:
df = (
    pos_intermediate
    .group_by('meal_code')
    .agg(
        pl.col('meal_type').first(),
        pl.col('co2').mean(),
    )
    .with_columns(
        pl.col('meal_type').fill_null("Not Mapped"),
        pl.col('co2').fill_null(dim_meals['co2'].mean())
    )

    .join(dim_meal_types, on='meal_type', how='left')
    .drop('meal_type', 'meal_type_en')
    .rename({'meal_type_id': 'meal_type'})
)
# df.head()

dishes_new = (
    dishes_new
    .join(df, on='meal_code', how='left')
    .with_row_index('id')

    .select(
        (pl.col('id') + 1 + dim_meals['id'].max()).cast(pl.Int64),
        pl.concat_list('meal_code').alias('meal_codes'),
        pl.concat_list('meal').alias('names'),
        'restaurants',
        'meal_type',
        pl.lit('25-26').alias('schoolyear'),
        pl.lit([]).alias('attributes'),
        pl.col('co2').cast(pl.Float32),
        pl.lit([]).alias('src'),
    )
)

dishes_new.head()


id,meal_codes,names,restaurants,meal_type,schoolyear,attributes,co2,src
i64,list[i64],list[str],list[i64],i64,str,list[null],f32,list[null]
869,[9313],"[""Luomutofua pestokastikkeessa""]","[2, 4]",7,"""25-26""",[],0.68,[]
870,[9308],"[""Kreikkaista pähkinä-pinaattikastiketta""]","[1, 2, 4]",7,"""25-26""",[],1.0625,[]
871,[9331],"[""Kebabmauste-kasvispyörykät & tomaatti-kebabkastike""]",[4],7,"""25-26""",[],0.604771,[]
872,[9339],"[""Afrikkalainen tomaattinen tagine""]",[1],4,"""25-26""",[],0.72,[]
873,[9328],"[""Härkis-BBQ-kastiketta""]","[2, 4]",7,"""25-26""",[],0.98,[]


Combine with existing `dim_meals`

In [11]:
dim_meals = pl.concat([dim_meals, dishes_new])

Save new `dim_meals`

In [12]:
path = "data/processed/dim_meals_1029.parquet"

dim_meals.write_parquet(path)